[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [16]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [1]:
import torch
import math

In [20]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
  Sq = Q.size()[-2]
  Sk = K.size()[-2]
  dk = K.size()[-1]
  # Better: Can use masked fill (attn_score.masked_fill(mask, float("-inf")))
  mask = torch.triu(torch.fill(torch.ones((Sq, Sk)), -math.inf), diagonal=1)
  attn_score = torch.softmax(Q@K.mT / math.sqrt(dk) + mask, dim=-1)
  print(attn_score)
  return attn_score @ V

In [21]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 5, 8)
V = torch.randn(1, 5, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4929, 0.5071, 0.0000, 0.0000, 0.0000],
         [0.6221, 0.0318, 0.3461, 0.0000, 0.0000],
         [0.2063, 0.6329, 0.1040, 0.0568, 0.0000]]])
Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [22]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5206, 0.4794, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1168, 0.4348, 0.4484, 0.0000, 0.0000, 0.0000],
         [0.5251, 0.2901, 0.1414, 0.0434, 0.0000, 0.0000],
         [0.1123, 0.3381, 0.1485, 0.3675, 0.0337, 0.0000],
         [0.3829, 0.0700, 0.0874, 0.0537, 0.0947, 0.3114]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5515, 0.4485, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3155, 0.0234, 0.6610, 0.0000, 0.0000, 0.0000],
         [0.2855, 0.1043, 0.5520, 0.0582, 0.0000, 0.0000],
         [0.0693, 0.0260, 0.3226, 0.0706, 0.5116, 0.0000],
         [0.1275, 0.0310, 0.7074, 0.0146, 0.0755, 0.0440]]])
  ✅ [1/4] Output shape (3.4ms)
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2422, 0.7578, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.347